# PDC Project — Phase 4 on Google Colab

This notebook runs the **hybrid training pipeline** (and optional Phase 4 configs) in a Linux environment with GPU if Colab assigns one.

**What you need**
- Your project on **GitHub** (recommended), *or* a **zip** of `pdc_project` uploaded to Colab.
- Runtime: **GPU** optional (faster); **CPU** works but slower.

**Not Docker**
- Colab runs Python directly in the VM. Docker is a separate reproducibility path on your PC; here we use the same `src/` and `configs/` without containers.

---
### Quick path
1. Set `REPO_URL` below (or use zip upload in the next section).
2. Run all cells top to bottom.
3. Download `logs.zip` from the last cell when experiments finish.

## 1) Optional: Google Drive (persistent storage)

Skip if you only need a one-off run. Drive helps if sessions disconnect.

In [ ]:
# Uncomment to mount Drive and optionally set WORK_ROOT to a Drive folder
# from google.colab import drive
# drive.mount('/content/drive')
# WORK_ROOT = '/content/drive/MyDrive/pdc_colab'
# import os; os.makedirs(WORK_ROOT, exist_ok=True); os.chdir(WORK_ROOT)

WORK_ROOT = '/content/pdc_workspace'
import os
os.makedirs(WORK_ROOT, exist_ok=True)
os.chdir(WORK_ROOT)
print('CWD:', os.getcwd())

## 2) Get the code: Git clone **or** upload zip

**Option A — GitHub (recommended)**  
Set `REPO_URL` to your repo HTTPS URL, then run the next cell.

**Option B — No GitHub**  
On your PC, zip the `pdc_project` folder. In Colab: *Files* → *Upload*, then run the “unzip” cell in comments.

In [ ]:
import os
import subprocess

# --- Option A: clone -------------------------------------------------
REPO_URL = 'https://github.com/YOUR_USERNAME/pdc_project.git'  # <-- EDIT THIS
BRANCH = 'main'  # or 'master'
PROJECT_DIR = os.path.join(WORK_ROOT, 'pdc_project')

if 'YOUR_USERNAME' in REPO_URL:
    raise RuntimeError('Edit REPO_URL to your real GitHub HTTPS URL, or use zip upload below.')

if os.path.isdir(PROJECT_DIR):
    print('Project folder exists, skipping clone:', PROJECT_DIR)
else:
    subprocess.run(
        ['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, PROJECT_DIR],
        check=True,
    )
    print('Cloned to', PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())

In [ ]:
# --- Option B: if you uploaded pdc_project.zip to /content/ ------------
# import zipfile, os
# os.chdir(WORK_ROOT)
# with zipfile.ZipFile('/content/pdc_project.zip', 'r') as z:
#     z.extractall(WORK_ROOT)
# os.chdir(os.path.join(WORK_ROOT, 'pdc_project'))
# print('CWD:', os.getcwd())

## 3) Install dependencies

Colab already ships with PyTorch. We install the rest from `requirements.txt`.  
If you see a torch version conflict, run the fallback cell after this one.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

In [ ]:
# Fallback only if torch import fails after pip install:
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

## 4) Sanity check (imports + project root)

Training scripts add the repo root to `sys.path` automatically when run as modules.

In [ ]:
import os, sys
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
from src.hybrid.controller import ModeController
print('ModeController OK, project root:', ROOT)

## 5) Smoke test — short hybrid run (optional)

Uses `configs/phase3_hybrid_safe.yaml` with default settings.  
On CPU this can take a long time; reduce `total_steps` in a copy of the YAML if needed.

**Tip:** For a faster smoke test, duplicate the YAML, set `total_steps: 50`, and point `CONFIG` there.

In [ ]:
import subprocess
import sys
import os

os.makedirs('logs', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('data', exist_ok=True)

CONFIG = 'configs/phase3_hybrid_safe.yaml'
subprocess.check_call([sys.executable, 'src/hybrid/launcher.py', '--nworkers', '2', '--config', CONFIG])

## 6) Phase 4 — single scenario (edit CONFIG)

Examples:
- `configs/phase4_sA_hybrid.yaml` — persistent straggler + hybrid  
- `configs/phase4_sB_hybrid.yaml` — transient straggler + hybrid  

Full matrix: run `experiments/run_scenarios.py` (next section).

In [ ]:
import subprocess
import sys

CONFIG = 'configs/phase4_sA_hybrid.yaml'
subprocess.check_call([sys.executable, 'src/hybrid/launcher.py', '--nworkers', '2', '--config', CONFIG])

## 7) Phase 4 — full scenario matrix (optional)

Runs all nine configs sequentially. **Can take many hours** on CPU; use GPU or run overnight.

For a subset: `--scenario sA` or `--scenario sA --mode hybrid`.

In [ ]:
import subprocess
import sys

# Full matrix, single seed
subprocess.check_call([sys.executable, 'experiments/run_scenarios.py'])

# Example: only Scenario A, all three modes
# subprocess.check_call([sys.executable, 'experiments/run_scenarios.py', '--scenario', 'sA'])

## 8) Threshold sweep (optional)

Grid search over controller thresholds; writes `logs/phase4_sweep_results.csv` (path may be under `logs/sweep/` per base config).

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, 'experiments/threshold_sweep.py'])

## 9) Aggregate results (tables for report)

Writes analysis to stdout; add `--output docs/phase4_results_tables.md` if `docs/` exists.

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, 'experiments/analyze_results.py',
    '--log-dir', './logs', '--tta-target', '40.0',
])

## 10) Download `logs/` and `checkpoints/` (zip)

Run after experiments finish. Files appear in Colab’s file browser; the zip downloads to your PC.

In [ ]:
import shutil
import os
import tempfile

ROOT = os.getcwd()
out_zip = os.path.join(ROOT, 'pdc_artifacts.zip')

with tempfile.TemporaryDirectory() as tmp:
    stage = os.path.join(tmp, 'artifacts')
    os.makedirs(stage, exist_ok=True)
    if os.path.isdir('logs'):
        shutil.copytree('logs', os.path.join(stage, 'logs'))
    if os.path.isdir('checkpoints'):
        shutil.copytree('checkpoints', os.path.join(stage, 'checkpoints'))
    shutil.make_archive(os.path.splitext(out_zip)[0], 'zip', root_dir=tmp, base_dir='artifacts')

print('Created:', out_zip, 'size MB:', round(os.path.getsize(out_zip) / 1e6, 2))

try:
    from google.colab import files
    files.download(out_zip)
    print('Download started (Colab).')
except ImportError:
    print('Not on Colab — zip saved at:', out_zip)

---
### Notes for your report

- **Environment:** Google Colab Linux VM, Python 3, optional GPU.
- **Same code** as local: `src/`, `configs/`, `experiments/`.
- **Docker** remains your local reproducibility story; Colab is an additional **cloud execution** path — state both clearly in Phase 5.